In [ ]:
import re
import pandas as pd
from pathlib import Path
from collections import defaultdict


# 1. Base configuration and path definitions
MAIN_FOLDER_PATH = Path(r"E:\DATA_FollowUp")
IMAGE_TYPES = ["CaSupp_25", "konv", "monoe_40kev"]

# Regex pattern to extract Patient ID and Follow-Up number from folder names
FOLDER_PATTERN = r"Myel_FollowUp_(\d+)_(\d+)"

# Hardcoded true acquisition dates for each patient's timeline
PATIENT_DATES = {
    "001": ["12-05-2022", "25-04-2023", "10-05-2024", "07-10-2025"],
    "002": ["07-10-2022", "21-06-2023", "28-06-2024", "14-02-2025", "04-07-2025"],
    "003": ["20-01-2023", "07-11-2023", "22-11-2024", "14-03-2025"],
    "004": ["29-11-2022", "20-11-2023", "22-02-2024", "06-06-2024", "24-06-2025"],
    "005": ["09-06-2022", "03-04-2023", "07-05-2024", "05-05-2025"],
    "006": ["27-03-2023", "26-06-2023", "08-11-2023", "27-02-2024", "04-06-2024"],
    "007": ["04-10-2023", "21-10-2024", "10-04-2025", "21-10-2025"],
    "008": ["23-10-2023", "20-02-2024", "14-08-2024", "01-04-2025"],
}

# Initialize data structure: {image_type: {patient_id: [(followup_num, file_path), ...]}}
patients_registry = {img: defaultdict(list) for img in IMAGE_TYPES}

# ==================================================================
# 2. SCANNING DIRECTORIES AND REGISTERING CSV FILES
# ==================================================================
if MAIN_FOLDER_PATH.exists():
    for folder in MAIN_FOLDER_PATH.iterdir():
        if not folder.is_dir():
            continue

        match = re.match(FOLDER_PATTERN, folder.name)
        if match:
            patient_id = match.group(1)
            followup_number = int(match.group(2))

            # Scan the inside of the valid patient folder
            for file in folder.iterdir():
                if file.is_file() and file.suffix == ".csv" and "spine_lesions" in file.name:
                    # Map the CSV file to its corresponding image type category
                    for img_type in IMAGE_TYPES:
                        if img_type in file.name:
                            patients_registry[img_type][patient_id].append((followup_number, file))

# ==================================================================
# 3. PROCESSING AND AGGREGATING LONGITUDINAL DATASETS
# ==================================================================
patient_follow_up_dfs = {}

for img_type, patient_data in patients_registry.items():
    patient_follow_up_dfs[img_type] = {}

    for patient_id, scans in patient_data.items():
        # Ensure chronological order based on the follow-up loop number
        scans_sorted = sorted(scans, key=lambda x: x[0])

        # Load and collect all dataframes for the current patient timeline
        loaded_dfs = []
        for followup, csv_path in scans_sorted:
            df = pd.read_csv(csv_path)
            loaded_dfs.append(df)

        # Merge all separate follow-up rows into a single timeline DataFrame
        aggregated_df = pd.concat(loaded_dfs, ignore_index=True)

        # Safely insert the acquisition date as the first column
        # Note: Assumes the number of discovered CSVs matches the PATIENT_DATES list length
        aggregated_df.insert(0, "Date", PATIENT_DATES[patient_id])

        # Store the finalized DataFrame into the main dictionary
        patient_follow_up_dfs[img_type][patient_id] = aggregated_df

In [ ]:
from scipy.stats import kendalltau
from statsmodels.stats.multitest import multipletests


# Initialize dictionary to store the final trend analysis data
kendall_results = {}
# ==================================================================
# TIME-SERIES TREND ANALYSIS (Kendall's Tau with FDR Correction)
# ==================================================================
for img_type, patient_data in patient_follow_up_dfs.items():
    kendall_results[img_type] = {}

    for patient_id, patient_df in patient_data.items():
        # Create a deep copy to prevent the SettingWithCopyWarning in Pandas
        df_cleaned = patient_df.copy()

        # Convert text dates to actual datetime objects and sort chronologically
        df_cleaned["Date"] = pd.to_datetime(df_cleaned["Date"], dayfirst=True)
        df_cleaned = df_cleaned.sort_values("Date").reset_index(drop=True)

        # Identify feature columns (Excluding the 'Date' column dynamically)
        feature_columns = [col for col in df_cleaned.columns if col != "Date"]

        # Create an ordinal time index (0, 1, 2...) representing sequential follow-ups
        time_index = range(len(df_cleaned))
        feature_trends = []

        # Compute the correlation of each feature against the timeline
        for feature in feature_columns:
            feature_values = df_cleaned[feature].values

            # Kendall's Tau calculation requires at least 3 data points to be meaningful
            if len(feature_values) < 3:
                continue

            tau_stat, p_value = kendalltau(time_index, feature_values)

            # Append raw results (handle cases where p_value might be NaN due to zero variance)
            feature_trends.append({
                "Feature": feature,
                "Tau": tau_stat,
                "P_value": p_value
            })

        # Convert the current patient's feature trends into a DataFrame
        results_df = pd.DataFrame(feature_trends)

        # Apply False Discovery Rate (FDR) multiple testing correction if data exists
        if not results_df.empty:
            # Drop any potential NaN p-values to prevent the correction algorithm from failing
            results_df = results_df.dropna(subset=["P_value"]).reset_index(drop=True)

            raw_p_values = results_df["P_value"]

            # Benjamini-Hochberg (fdr_bh) correction method
            _, fdr_p_values, _, _ = multipletests(raw_p_values, method="fdr_bh")
            results_df["FDR_p"] = fdr_p_values

            # Sort the final table by the strongest raw correlation significance
            results_df = results_df.sort_values("P_value").reset_index(drop=True)

        # Store the finalized trends table back into the registry
        kendall_results[img_type][patient_id] = results_df

# Output the multi-indexed dictionary containing all calculated trends
kendall_results

In [ ]:
import matplotlib.pyplot as plt


# 1. Setup global plotting style and configuration
plt.style.use('default')

# Define target image type and specific radiomics feature for visualization
image_type = "monoe_40kev"
feature = "original_glcm_DifferenceAverage"

# Initialize the figure size for longitudinal line plot
plt.figure(figsize=(8, 6))
# ==================================================================
# 2. PLOTTING PATIENT TIMELINES (Months from Baseline)
# ==================================================================
for patient_id, patient_df in patient_follow_up_dfs[image_type].items():
    # Create a deep copy to safely perform transformations
    df_cleaned = patient_df.copy()

    # Ensure datetime format and chronological order
    df_cleaned["Date"] = pd.to_datetime(df_cleaned["Date"], dayfirst=True)
    df_cleaned = df_cleaned.sort_values("Date").reset_index(drop=True)

    # Calculate timedelta in months relative to each patient's first exam (baseline)
    # Using average month length (30.44 days) for accurate scaling
    baseline_date = df_cleaned["Date"].iloc[0]
    delta_months = (df_cleaned["Date"] - baseline_date).dt.days / 30.44

    # Plot the individual patient's trajectory line
    plt.plot(
        delta_months,
        df_cleaned[feature],
        marker="o",
        linestyle="-",
        linewidth=1.5,
        label=f"Patient {patient_id}"
    )

# ==================================================================
# 3. GRAPH STYLING AND RENDERING
# ==================================================================
plt.xlabel("Months from baseline", fontsize=11)
plt.ylabel(feature, fontsize=11)
plt.title(f"All Patients Trajectory - {feature}\nImage Type: {image_type}", fontsize=13)

# Add elements to improve scannability
plt.legend(title="Patient ID", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle="--", alpha=0.5)

# Adjust layout to accommodate the legend shifted outside the plot area
plt.tight_layout()
plt.show()